In [ ]:
import os, shutil

# Point this to wherever your brain tumor dataset lives
SOURCE = "/content/drive/MyDrive/BrainTumer_Detection/brain-tumor-mri-dataset"   # adjust path
DEST   = "/content/drive/MyDrive/BrainTumer_Detection/gatekeeper_dataset/brain_mri"

os.makedirs(DEST, exist_ok=True)

# It will walk all subfolders (glioma, meningioma, etc.) and copy images
count = 0
for root, dirs, files in os.walk(SOURCE):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            shutil.copy(os.path.join(root, f), os.path.join(DEST, f"{count}_{f}"))
            count += 1

print(f"Copied {count} brain MRI images")

In [ ]:
# Run this first — upload your kaggle.json API key
from google.colab import files
files.upload()   # select your kaggle.json file when prompted

# Move it to the right place
import os
os.makedirs("/root/.config/kaggle", exist_ok=True)
os.system("mv kaggle.json /root/.config/kaggle/kaggle.json")
os.system("chmod 600 /root/.config/kaggle/kaggle.json")

# Verify
os.system("kaggle datasets list")
print("Kaggle API ready ✅")

In [ ]:
DEST = "/content/drive/MyDrive/BrainTumer_Detection/gatekeeper_dataset/non_brain_mri"
os.makedirs(DEST, exist_ok=True)
print(f"Destination ready: {DEST}")

In [ ]:
import shutil
from PIL import Image
from pathlib import Path

# Download
os.makedirs("/content/chest_xray_raw", exist_ok=True)
os.system("kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p /content/chest_xray_raw --unzip")

# Collect all images from all subfolders
chest_source = Path("/content/chest_xray_raw")
all_chest = list(chest_source.rglob("*.jpeg")) + list(chest_source.rglob("*.jpg")) + list(chest_source.rglob("*.png"))

print(f"Total chest X-ray images found: {len(all_chest)}")

# Copy 3500 randomly
import random
random.seed(42)
selected = random.sample(all_chest, min(3000, len(all_chest)))

count = 0
for src in selected:
    try:
        img = Image.open(src).convert("RGB").resize((224, 224))
        img.save(f"{DEST}/chest_{count:04d}.jpg")
        count += 1
    except Exception as e:
        print(f"Skipped {src.name}: {e}")

print(f"✅ Chest X-Ray: {count} images saved")

In [ ]:
import tensorflow_datasets as tfds
import numpy as np

# Load CIFAR-10 — no Kaggle needed, downloads automatically
ds = tfds.load("cifar10", split="train", as_supervised=True)

count = 0
TARGET = 3000

for img_tensor, label in ds.take(TARGET):
    try:
        img_array = img_tensor.numpy()
        img = Image.fromarray(img_array).resize((224, 224))
        img.save(f"{DEST}/cifar_{count:04d}.jpg")
        count += 1
    except Exception as e:
        print(f"Skipped frame {count}: {e}")

print(f"✅ CIFAR-10: {count} images saved")

In [ ]:
all_saved = list(Path(DEST).glob("*.jpg"))
total = len(all_saved)

chest_count = len([f for f in all_saved if f.name.startswith("chest")])
cifar_count = len([f for f in all_saved if f.name.startswith("cifar")])

print("=" * 40)
print(f"  Chest X-Ray   : {chest_count:>6} images")
print(f"  CIFAR-10      : {cifar_count:>6} images")
print(f"  {'─'*25}")
print(f"  TOTAL         : {total:>6} images")
print("=" * 40)

if total >= 5000:
    print("✅ Dataset ready — proceed to training")
else:
    print(f"⚠️  Only {total} images — check for download errors above")

In [ ]:
import os

dataset_path = "/content/drive/MyDrive/BrainTumer_Detection/gatekeeper_dataset"

if os.path.exists(dataset_path):
    folders = [f for f in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, f))]
    print(f"Summary of folders in: {dataset_path}\n")
    for folder in sorted(folders):
        folder_path = os.path.join(dataset_path, folder)
        files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"  {folder}: {len(files)} images")
else:
    print(f"Path not found: {dataset_path}")

In [ ]:
import shutil, os

print("Copying dataset to local storage...")

shutil.copytree(
    "/content/drive/MyDrive/BrainTumer_Detection/gatekeeper_dataset",
    "/content/gatekeeper_data",
    ignore=shutil.ignore_patterns(".ipynb_checkpoints", "*.h5", "*.png"),
)

# Verify
brain     = len(os.listdir("/content/gatekeeper_data/brain_mri"))
non_brain = len(os.listdir("/content/gatekeeper_data/non_brain_mri"))
print(f"✅ Copy complete!")
print(f"   brain_mri     : {brain} images")
print(f"   non_brain_mri : {non_brain} images")

In [ ]:
from PIL import Image
import os
from pathlib import Path

DATA_DIR = "/content/gatekeeper_data"
removed  = 0
kept     = 0

print("Scanning for corrupted images...")

for img_path in Path(DATA_DIR).rglob("*"):
    if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
        continue
    try:
        with Image.open(img_path) as img:
            img.verify()   # checks if file is valid
        kept += 1
    except Exception:
        print(f"  ❌ Removing corrupted: {img_path.name}")
        os.remove(img_path)
        removed += 1

print(f"\n✅ Scan complete")
print(f"   Kept    : {kept}")
print(f"   Removed : {removed}")

# Final count per class
for folder in ["brain_mri", "non_brain_mri"]:
    count = len(list(Path(f"{DATA_DIR}/{folder}").glob("*")))
    print(f"   {folder} : {count} images")

In [ ]:
import tensorflow_datasets as tfds
from PIL import Image
import os

DEST = "/content/gatekeeper_data/non_brain_mri"

# ── Add real photographs from multiple TFDS sources ───────────────────────────

# 1. Cats and dogs — real photographs
print("Loading cats_vs_dogs...")
ds1 = tfds.load("cats_vs_dogs", split="train", as_supervised=True)
count = 0
for img, _ in ds1.take(2000):
    try:
        pil = Image.fromarray(img.numpy()).convert("RGB").resize((224, 224))
        pil.save(f"{DEST}/catdog_{count:04d}.jpg")
        count += 1
    except: pass
print(f"  ✅ Added {count} cat/dog photos")

# 2. Colorectal histology — real medical but NOT brain MRI
print("Loading colorectal_histology...")
ds2 = tfds.load("colorectal_histology", split="train", as_supervised=True)
count = 0
for img, _ in ds2.take(1000):
    try:
        pil = Image.fromarray(img.numpy()).convert("RGB").resize((224, 224))
        pil.save(f"{DEST}/histo_{count:04d}.jpg")
        count += 1
    except: pass
print(f"  ✅ Added {count} histology images")

# 3. Food101 — everyday photos
print("Loading food101...")
ds3 = tfds.load("food101", split="train", as_supervised=True)
count = 0
for img, _ in ds3.take(1500):
    try:
        pil = Image.fromarray(img.numpy()).convert("RGB").resize((224, 224))
        pil.save(f"{DEST}/food_{count:04d}.jpg")
        count += 1
    except: pass
print(f"  ✅ Added {count} food photos")

# ── Final count ───────────────────────────────────────────────────────────────
total = len(os.listdir(DEST))
print(f"\nTotal non_brain_mri images : {total}")

In [ ]:
import os, shutil
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

# ── Delete .ipynb_checkpoints if exists ───────────────────────────────────────
CHECKPOINTS = "/content/drive/MyDrive/BrainTumer_Detection/gatekeeper_dataset/.ipynb_checkpoints"
if os.path.exists(CHECKPOINTS):
    shutil.rmtree(CHECKPOINTS)
    print("✅ Deleted .ipynb_checkpoints")

# Verify only 2 folders exist
folders = os.listdir("/content/drive/MyDrive/BrainTumer_Detection/gatekeeper_dataset")
print(f"Folders found: {folders}")
assert len(folders) == 2, f"❌ Expected 2 folders, found {len(folders)} — remove extra folders first!"

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR    = "/content/gatekeeper_data"
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
EPOCHS_HEAD = 3
EPOCHS_FINE = 3
SAVE_PATH   = "/content/drive/MyDrive/BrainTumer_Detection/gatekeeper_dataset/gatekeeper_model.h5"

# ── Data generators ───────────────────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
)

train_gen = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training",
    classes=["brain_mri", "non_brain_mri"],   # brain_mri=0, non_brain_mri=1
)

val_gen = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation",
    classes=["brain_mri", "non_brain_mri"],
)

print("\nClass indices:", train_gen.class_indices)
# Must show: {'brain_mri': 0, 'non_brain_mri': 1}

# ── Build model ───────────────────────────────────────────────────────────────
base = MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base.trainable = False

inputs  = tf.keras.Input(shape=(*IMG_SIZE, 3))
x       = base(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dropout(0.3)(x)
x       = layers.Dense(128, activation="relu")(x)
x       = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

gatekeeper = models.Model(inputs, outputs)
gatekeeper.summary()

# ── Phase 1: Train head only ──────────────────────────────────────────────────
gatekeeper.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
)

print("\n=== Phase 1: Training head only ===")
history1 = gatekeeper.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_HEAD,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=3,
            restore_best_weights=True,
            verbose=1,
        ),
    ]
)

# ── Phase 2: Fine-tune top 30 layers of MobileNetV2 ──────────────────────────
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

print(f"\nTrainable layers in phase 2: {sum(1 for l in gatekeeper.layers if l.trainable)}")

gatekeeper.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
)

print("\n=== Phase 2: Fine-tuning top layers ===")
history2 = gatekeeper.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_FINE,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=3,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            SAVE_PATH,
            save_best_only=True,
            monitor="val_auc",
            verbose=1,
        ),
    ]
)

print(f"\n✅ Model saved to {SAVE_PATH}")

# ── Plot training history ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Combine both phases
acc     = history1.history["accuracy"]     + history2.history["accuracy"]
val_acc = history1.history["val_accuracy"] + history2.history["val_accuracy"]
auc     = history1.history["auc"]          + history2.history["auc"]
val_auc = history1.history["val_auc"]      + history2.history["val_auc"]
phase_boundary = len(history1.history["accuracy"])

axes[0].plot(acc,     label="Train Accuracy")
axes[0].plot(val_acc, label="Val Accuracy")
axes[0].axvline(phase_boundary, color="red", linestyle="--", label="Fine-tune start")
axes[0].set_title("Accuracy")
axes[0].legend()

axes[1].plot(auc,     label="Train AUC")
axes[1].plot(val_auc, label="Val AUC")
axes[1].axvline(phase_boundary, color="red", linestyle="--", label="Fine-tune start")
axes[1].set_title("AUC")
axes[1].legend()

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/BrainTumer_Detection/gatekeeper_dataset/training_plot.png")
plt.show()
print("✅ Training plot saved")

# ── Final validation score ────────────────────────────────────────────────────
loss, acc, auc = gatekeeper.evaluate(val_gen, verbose=0)
print(f"\nFinal Validation Results:")
print(f"  Accuracy : {acc*100:.2f}%")
print(f"  AUC      : {auc:.4f}")
print(f"  Loss     : {loss:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from PIL import Image
from tensorflow.keras.models import load_model

# Load the gatekeeper model
gatekeeper = load_model("/content/drive/MyDrive/BrainTumer_Detection/gatekeeper_dataset/gatekeeper_model.h5")

# ── Config ────────────────────────────────────────────────────────────────────
IMG_SIZE  = (224, 224)
THRESHOLD = 0.60   # below = brain MRI, above = not brain MRI

# ── Single image test function ────────────────────────────────────────────────
def is_brain_mri(image_path, threshold=THRESHOLD, show=True):
    img = load_img(image_path, target_size=IMG_SIZE)
    arr = img_to_array(img) / 255.0
    arr = np.expand_dims(arr, axis=0)

    prob     = gatekeeper.predict(arr, verbose=0)[0][0]
    is_mri   = prob < threshold
    label    = "✅ Valid Brain MRI" if is_mri else "❌ Not a Brain MRI"
    color    = "green" if is_mri else "red"
    conf     = (1 - prob) * 100 if is_mri else prob * 100

    print(f"  File     : {image_path}")
    print(f"  Score    : {prob:.4f}  (threshold: {threshold})")
    print(f"  Result   : {label}")
    print(f"  Confidence : {conf:.1f}%")
    print()

    if show:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        fig.patch.set_facecolor("#0f172a")

        # Left — image
        original = Image.open(image_path).convert("RGB")
        axes[0].imshow(original)
        axes[0].set_title(label, color=color, fontsize=13, fontweight="bold", pad=10)
        axes[0].axis("off")

        # Right — probability bar
        axes[1].set_facecolor("#1e293b")
        bar_colors = ["#22c55e", "#ef4444"]   # green for brain, red for not
        values     = [(1 - prob) * 100, prob * 100]
        bar_labels = ["Brain MRI", "Not Brain MRI"]
        bars = axes[1].barh(bar_labels, values, color=bar_colors, height=0.4)
        axes[1].set_xlim(0, 100)
        axes[1].set_xlabel("Confidence %", color="white")
        axes[1].tick_params(colors="white")
        axes[1].spines[:].set_color("#334155")
        for bar, val in zip(bars, values):
            axes[1].text(val + 1, bar.get_y() + bar.get_height()/2,
                         f"{val:.1f}%", va="center", color="white", fontsize=11)

        # Threshold line
        axes[1].axvline(threshold * 100, color="yellow", linestyle="--",
                        linewidth=1.2, label=f"Threshold ({threshold*100:.0f}%)")
        axes[1].legend(facecolor="#1e293b", labelcolor="white", fontsize=9)
        axes[1].set_title("Class Probabilities", color="white", fontsize=11, pad=10)

        plt.suptitle(
            f"NeuroPath Gatekeeper",
            color="white", fontsize=14, fontweight="bold", y=1.01
        )
        plt.tight_layout()
        plt.show()

    return is_mri, prob


# ── Test multiple images at once ──────────────────────────────────────────────
def test_batch(image_paths, threshold=THRESHOLD):
    print("=" * 55)
    print("  NEUROPATH GATEKEEPER — BATCH TEST")
    print("=" * 55)

    results = []
    for path in image_paths:
        try:
            is_mri, prob = is_brain_mri(path, threshold=threshold, show=False)
            results.append((path, is_mri, prob))
        except Exception as e:
            print(f"  ⚠️  Could not load {path}: {e}\n")

    # Summary table
    print("=" * 55)
    print(f"  {'Image':<25} {'Score':>7}  Result")
    print(f"  {'-'*25}  {'-'*7}  {'-'*20}")
    for path, is_mri, prob in results:
        name   = path.split("/")[-1][:24]
        status = "✅ Brain MRI" if is_mri else "❌ Not Brain MRI"
        print(f"  {name:<25} {prob:>7.4f}  {status}")
    print("=" * 55)

    passed = sum(1 for _, is_mri, _ in results if is_mri)
    print(f"  Passed gatekeeper : {passed}/{len(results)}")
    print("=" * 55)
    return results


# ── Run tests ─────────────────────────────────────────────────────────────────

# Single image with visual output
is_brain_mri("/content/WIN_20260424_13_56_25_Pro.jpg")
# is_brain_mri("/content/mri image.jpg")
# is_brain_mri("/content/bones mri.jpeg")
# is_brain_mri("/content/mri brain.jpeg")

# OR test multiple images at once without individual plots
test_batch([
    "/content/WIN_20260424_13_56_25_Pro.jpg",

    # add more paths here
])